# Test 02: Core Adaptive Concentric-Ring Grid Engine
Implementing a purely spatial `MultiResGrid` that aligns all cells to a single ego-centric origin, partitions points into three concentric rings by radial distance, and rasterizes each ring at a specific resolution.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import open3d as o3d
import matplotlib.pyplot as plt

# Load PCD
PCD_FILE = "../../data/raw/lidar_with_intensity_and_clusters/frame_00008.pcd"
#PCD_FILE="frame_0031.pcd"
if not os.path.exists(PCD_FILE):
    print(f"Error: {PCD_FILE} not found.")
else:
    t_pcd = o3d.t.io.read_point_cloud(PCD_FILE)
    points = t_pcd.point.positions.numpy()
    
    # Filter valid points
    valid_mask = np.isfinite(points).all(axis=1)
    valid_points = points[valid_mask]
    
    # Get intensity
    if 'intensity' in t_pcd.point:
        intensities = t_pcd.point.intensity.numpy()[valid_mask].flatten()
    else:
        intensities = np.zeros(len(valid_points))
        
    print(f"Loaded {len(valid_points)} valid points.")


Loaded 52878 valid points.


In [2]:
class MultiResGrid:
    def __init__(self, ring_bounds=(50.0, 85.0, 100.0), cell_sizes=(0.05, 0.25, 0.50)):
        self.ring_bounds = np.array(ring_bounds, dtype=float)
        self.cell_sizes = np.array(cell_sizes, dtype=float)
        self.n_rings = len(ring_bounds)
        
    def assign_ring(self, r):
        # Assign points to ring index based on distance r
        # np.searchsorted with side='left':
        # r <= 50 -> idx=0
        # 50 < r <= 85 -> idx=1
        # 85 < r <= 100 -> idx=2
        # r > 100 -> idx=3
        idx = np.searchsorted(self.ring_bounds, r, side='left')
        idx[idx >= self.n_rings] = -1
        return idx
        
    def rasterize(self, xyz, intensity):
        x = xyz[:, 0]
        y = xyz[:, 1]
        z = xyz[:, 2]
        
        # 1. Calculate r from x and y
        r = np.sqrt(x**2 + y**2)
        
        # 2. Assign each point to exactly one ring
        ring_idx = self.assign_ring(r)
        
        ring_representations = {}
        metrics = []
        
        total_input_points = len(xyz)
        total_occupied_cells = 0
        
        # Equivalent uniform 5 cm grid covering ±100 m
        uniform_cell_area = 0.05 * 0.05
        uniform_total_area = (2 * 100)**2
        equivalent_uniform_cells = int(uniform_total_area / uniform_cell_area)
        
        for k in range(self.n_rings):
            mask = ring_idx == k
            x_k, y_k, z_k, int_k = x[mask], y[mask], z[mask], intensity[mask]
            
            cs = self.cell_sizes[k]
            bound = self.ring_bounds[k]
            prev_bound = self.ring_bounds[k-1] if k > 0 else 0.0
            
            # Theoretical dense grid memory/cell count for this ring
            ring_area = np.pi * (bound**2 - prev_bound**2)
            dense_cells = int(np.ceil(ring_area / (cs * cs)))
            
            if not mask.any():
                metrics.append({
                    'ring_index': k,
                    'cell_size': cs,
                    'outer_bound': bound,
                    'dense_cells': dense_cells,
                    'occupied_cell_count': 0,
                    'input_points': 0
                })
                continue
                
            # 3. Calculate global XY cell indices
            ix = np.floor(x_k / cs).astype(int)
            iy = np.floor(y_k / cs).astype(int)
            
            df = pd.DataFrame({
                'z': z_k,
                'intensity': int_k,
                'ix': ix,
                'iy': iy
            })
            
            # 4. Identify occupied cells and store height, count, occupancy, representative intensity
            grouped = df.groupby(['ix', 'iy'])
            
            agg = grouped.agg(
                z_mean=('z', 'mean'),
                z_min=('z', 'min'),
                z_max=('z', 'max'),
                point_count=('z', 'count'),
                representative_intensity=('intensity', 'max')
            ).reset_index()
            
            agg['occupancy'] = 1
            
            # Calculate representative x, y back from ix, iy for visualization
            agg['x_center'] = (agg['ix'] + 0.5) * cs
            agg['y_center'] = (agg['iy'] + 0.5) * cs
            
            ring_representations[k] = agg
            
            occ_count = len(agg)
            total_occupied_cells += occ_count
            
            metrics.append({
                'ring_index': k,
                'cell_size': cs,
                'outer_bound': bound,
                'dense_cells': dense_cells,
                'occupied_cell_count': occ_count,
                'input_points': len(x_k)
            })
            
        reduction = (total_input_points - total_occupied_cells) / total_input_points * 100 if total_input_points > 0 else 0
        
        return ring_representations, metrics, total_input_points, total_occupied_cells, reduction, equivalent_uniform_cells


In [3]:
if 'valid_points' in locals():
    grid = MultiResGrid()
    ring_reps, metrics, total_in, total_occ, reduction, uniform_cells = grid.rasterize(valid_points, intensities)
    
    print("="*60)
    print("MultiResGrid Rasterization Metrics")
    print("="*60)
    print(f"Total Input Points: {total_in}")
    print(f"Total Occupied Cells: {total_occ}")
    print(f"Percentage Reduction: {reduction:.2f}%")
    print(f"Equivalent uniform 5cm grid (±100m) theoretical cells: {uniform_cells:,}")
    
    total_adaptive_dense = sum([m['dense_cells'] for m in metrics])
    print(f"Total Adaptive grid theoretical dense cells: {total_adaptive_dense:,}")
    print(f"Memory reduction vs Uniform Dense: {(uniform_cells - total_adaptive_dense) / uniform_cells * 100:.2f}%")
    print("-" * 60)
    
    for m in metrics:
        print(f"Ring {m['ring_index']} (<= {m['outer_bound']}m, cell={m['cell_size']}m):")
        print(f"  Input Points: {m['input_points']}")
        print(f"  Occupied Cells: {m['occupied_cell_count']:,}")
        print(f"  Theoretical Dense Cells: {m['dense_cells']:,}")
        print()


MultiResGrid Rasterization Metrics
Total Input Points: 52878
Total Occupied Cells: 11426
Percentage Reduction: 78.39%
Equivalent uniform 5cm grid (±100m) theoretical cells: 15,999,999
Total Adaptive grid theoretical dense cells: 3,413,970
Memory reduction vs Uniform Dense: 78.66%
------------------------------------------------------------
Ring 0 (<= 50.0m, cell=0.05m):
  Input Points: 52878
  Occupied Cells: 11,426
  Theoretical Dense Cells: 3,141,593

Ring 1 (<= 85.0m, cell=0.25m):
  Input Points: 0
  Occupied Cells: 0
  Theoretical Dense Cells: 237,505

Ring 2 (<= 100.0m, cell=0.5m):
  Input Points: 0
  Occupied Cells: 0
  Theoretical Dense Cells: 34,872



In [4]:
if 'ring_reps' in locals():
    all_pcds = []
    
    # Assign distinct colormaps for each ring to visually verify them
    cmaps = ['autumn', 'winter', 'summer']
    
    total_vis_points = 0
    for k, df in ring_reps.items():
        if len(df) == 0:
            continue
            
        pts = df[['x_center', 'y_center', 'z_mean']].values
        total_vis_points += len(pts)
        
        # Color by intensity but scaled to the distinct colormap for this ring
        intensities_k = df['representative_intensity'].values
        
        ptp = intensities_k.max() - intensities_k.min()
        if ptp > 0:
            intensity_norm = (intensities_k - intensities_k.min()) / ptp
        else:
            intensity_norm = np.zeros_like(intensities_k)
            
        cmap = plt.get_cmap(cmaps[k % len(cmaps)])
        colors = cmap(intensity_norm)[:, :3]
        
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(pts)
        pcd.colors = o3d.utility.Vector3dVector(colors)
        
        all_pcds.append(pcd)
        
    print(f"Visualizing {total_vis_points} cells across {len(all_pcds)} rings...")
    print("Colors:")
    print("  Ring 0 (0-50m):   Autumn (Red-Yellow)")
    print("  Ring 1 (50-85m):  Winter (Blue-Green)")
    print("  Ring 2 (85-100m): Summer (Green-Yellow)")
    
    if all_pcds:
        o3d.visualization.draw_geometries(all_pcds, 
                                          window_name="MultiResGrid Concentric Rings", 
                                          width=1280, 
                                          height=720)


Visualizing 11426 cells across 1 rings...
Colors:
  Ring 0 (0-50m):   Autumn (Red-Yellow)
  Ring 1 (50-85m):  Winter (Blue-Green)
  Ring 2 (85-100m): Summer (Green-Yellow)
